# Notebook 01 — Agentic workflows: one step vs many steps

**Time:** about 30 minutes

**What you will learn**
- What an *agentic workflow* is
- How to break a task into small steps (*task decomposition*)
- How to compare a one-step answer with a multi-step answer

### The big idea
- **One step (not agentic):** "Write an essay" → the model writes it in one go, with no chance to plan or fix mistakes.
- **Many steps (agentic):** plan → write → check → improve. Each step is a small, simple request.
- It is like a student who writes an outline, a first draft, asks for feedback, and then improves the draft.

### ▶️ Run the two setup cells below first
They check that Ollama is ready and create `ask()`, the one helper we use to talk to the model.

In [1]:
# STEP 0 — Is Ollama running? Is the model downloaded?
import ollama

MODEL = "llama3.2"     # local model (bigger one: "qwen2.5:7b")

try:
    installed = [m.model for m in ollama.list().models]
    print("Models on this computer:", installed)
    if MODEL in installed or MODEL + ":latest" in installed:
        print("✅ Ready to go with", MODEL)
    else:
        print(f"⚠️ {MODEL} is missing. Run: ollama pull {MODEL}")
except ConnectionError:
    print("⚠️ Ollama is not running. Start it, then run this again:")
    print("   Arch Linux: sudo systemctl start ollama")
    print("   Windows 11: open Ollama from the Start menu")

Models on this computer: ['llama3.2:latest']
✅ Ready to go with llama3.2


In [2]:
# STEP 1 — A helper: send a message, get the answer back as text
DEFAULT_ROLE = "You are a helpful assistant. Keep answers short."

def ask(prompt, system=DEFAULT_ROLE):
    reply = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        options={"temperature": 0},    # 0 = more predictable
    )
    return reply.message.content

print("ask() is ready.")

ask() is ready.


## Part 1 — The one-step way
We ask for the whole essay at once and measure the time.

In [3]:
import time

TOPIC = "How AI can help university students"

start = time.time()
one_step_essay = ask(f"Write a 150-word essay about: {TOPIC}")
one_step_seconds = time.time() - start

print(one_step_essay)
print(f"\n⏱ {one_step_seconds:.1f} seconds, 1 model call")

The integration of Artificial Intelligence (AI) in universities can significantly enhance the learning experience for students. AI-powered tools can help personalize education by providing tailored recommendations for course materials, assignments, and study plans. AI-driven chatbots can also offer 24/7 support, answering students' queries and providing guidance on academic matters. Additionally, AI-based analytics can help identify areas where students need extra support, enabling instructors to provide targeted interventions.

AI can also facilitate remote learning, enabling students to access course materials and interact with instructors from anywhere. Furthermore, AI-powered virtual teaching assistants can help with administrative tasks, freeing up instructors to focus on teaching and mentoring. Overall, AI has the potential to revolutionize the way universities deliver education, making it more accessible, efficient, and effective. By harnessing the power of AI, universities can 

## Part 2 — The agentic way: four small steps

| Step | What we ask the model |
|---|---|
| 1. Plan | Write a short outline |
| 2. Draft | Write the essay from the outline |
| 3. Check | List the weaknesses of the draft |
| 4. Improve | Rewrite the draft using the feedback |

Each step uses the **output of the step before**. Read each result as it appears.

In [4]:
start = time.time()

outline = ask(f"Write a 4-point essay outline about: {TOPIC}")
print("STEP 1 — OUTLINE\n", outline, "\n")

STEP 1 — OUTLINE
 Here is a 4-point essay outline on "How AI can help university students":

I. Introduction
  A. Brief overview of AI and its growing presence in education
  B. Thesis statement: AI can significantly enhance the learning experience for university students

II. Personalized Learning and Adaptation
  A. AI-powered adaptive learning systems
  B. Tailored learning plans and content delivery
  C. Improved student outcomes and engagement

III. Intelligent Tutoring and Support
  A. AI-based virtual learning assistants
  B. 24/7 support and guidance
  C. Enhanced student success and well-being

IV. Data-Driven Insights and Feedback
  A. AI-powered analytics and feedback tools
  B. Data-driven insights for improved teaching and learning
  C. Enhanced student assessment and evaluation 



In [5]:
draft = ask("Write a 150-word essay that follows this outline:\n"
            + outline)
print("STEP 2 — DRAFT\n", draft, "\n")

STEP 2 — DRAFT
 The integration of Artificial Intelligence (AI) in education is revolutionizing the way university students learn. AI can significantly enhance the learning experience by providing personalized, intelligent, and data-driven support. AI-powered adaptive learning systems tailor learning plans and content delivery to individual students' needs, leading to improved outcomes and engagement. Virtual learning assistants, powered by AI, offer 24/7 support and guidance, enhancing student success and well-being. Additionally, AI-powered analytics and feedback tools provide data-driven insights for improved teaching and learning, enabling educators to refine their methods and assess student performance more effectively. By leveraging AI, universities can create a more inclusive, effective, and efficient learning environment, ultimately benefiting students and driving academic success. 



In [6]:
feedback = ask("List the 3 biggest weaknesses of this essay. "
               "Be specific.\n\nESSAY:\n" + draft)
print("STEP 3 — FEEDBACK\n", feedback, "\n")

STEP 3 — FEEDBACK
 Here are the 3 biggest weaknesses of the essay:

1. Lack of concrete examples: The essay makes general claims about AI's benefits, but lacks specific examples or case studies to support these claims. This makes it difficult to fully understand the impact of AI on education.

2. Overly broad statements: Phrases like "revolutionizing the way university students learn" and "creating a more inclusive, effective, and efficient learning environment" are overly broad and lack concrete evidence to back them up.

3. No consideration of potential drawbacks: The essay focuses solely on the benefits of AI in education, without discussing potential drawbacks or limitations, such as job displacement, bias in AI decision-making, or unequal access to AI-powered resources. 



In [7]:
final_essay = ask("Improve the essay using the feedback. "
                  "Keep it about 150 words.\n\n"
                  f"ESSAY:\n{draft}\n\nFEEDBACK:\n{feedback}")
workflow_seconds = time.time() - start

print("STEP 4 — FINAL ESSAY\n", final_essay)

STEP 4 — FINAL ESSAY
 Here's an improved version of the essay, within the 150-word limit:

The integration of Artificial Intelligence (AI) in education is transforming the learning experience for university students. AI-powered adaptive learning systems provide personalized support, tailoring learning plans and content delivery to individual students' needs. Virtual learning assistants, powered by AI, offer 24/7 guidance and support, enhancing student success and well-being. AI-powered analytics and feedback tools provide data-driven insights for improved teaching and learning, enabling educators to refine their methods and assess student performance more effectively.

However, it's essential to consider the potential drawbacks of AI in education. For instance, AI-powered systems can perpetuate biases and exacerbate existing inequalities. Moreover, the increasing reliance on AI may displace certain teaching roles. To mitigate these risks, universities must prioritize responsible AI dev

## Part 3 — Compare the two results

In [8]:
words_1 = len(one_step_essay.split())
words_4 = len(final_essay.split())
t1 = round(one_step_seconds, 1)
t4 = round(workflow_seconds, 1)

print(f"One step : {words_1} words, 1 call,  {t1} seconds")
print(f"Workflow : {words_4} words, 4 calls, {t4} seconds")

One step : 139 words, 1 call,  18.0 seconds
Workflow : 147 words, 4 calls, 107.3 seconds


👉 **Discuss:** which essay is better organised? Was the extra time worth it?

The workflow is **slower** (more calls) but usually **better**, because every step can be checked and improved.
This trade-off — *quality vs time* — appears in every notebook of this workshop.

## Part 4 — Let the model break a task into steps

A good habit: ask *"How would a careful person do this, step by step?"*
Here the model does the breaking-down for us.

In [9]:
task = "Plan a one-day study trip for 30 students to a museum."

steps = ask("Break this task into 5 short numbered steps. "
            "One line per step.\n\nTASK: " + task)
print(steps)

Here are the 5 short steps:

1. Choose a museum that aligns with the students' interests and curriculum.
2. Plan the itinerary, including arrival and departure times, to ensure a smooth and efficient day.
3. Arrange for transportation to and from the museum, and consider booking a guided tour if available.
4. Prepare educational materials and activities to enhance the students' learning experience.
5. Confirm the details with the museum and students, and have a backup plan in case of unexpected changes.


## Part 5 — How much freedom does the AI get? (degrees of autonomy)

| Level | Who decides the steps? | Example | Predictable? |
|---|---|---|---|
| Low | **You** write every step in code | Part 2 above | Very |
| Medium | You fix the stages; the model makes small choices | A research helper that picks search words | Mostly |
| High | The **model** decides which steps to take | A planning agent (Notebook 05) | Less |

More freedom = more flexible, but harder to predict and to test.

## ✍️ Try it yourself
1. Change `TOPIC` in Part 1 and run Parts 1–3 again.
2. Add a fifth step to the workflow: *"Check the essay has a clear conclusion"*.

## ✅ Summary
- An **agentic workflow** = several small model calls, each using the previous result.
- It is slower than one call, but each step can be checked and improved.
- Breaking a task into steps is the first skill of building agents.

**Next:** Notebook 02 — *reflection*: letting the model check and fix its own work.